Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import os
import shutil

# Step 1: Ensure 'data' directory exists
if not os.path.exists("data"):
    os.makedirs("data")

# Step 2: Run the SQL query and get the results as a DataFrame
query = """
WITH t AS (
    SELECT 
        cast(json_extract(dt.metadata, '$.correlation_id') AS string) AS chat_reference_id, 
        result,
        us.name AS annotatedBy,
        rs.annotated_at AS annotatedAt,
        dt.ticket_id AS ticket_id
    FROM (
        SELECT 
            id,
            data_id,
            annotated_by,
            annotated_at,
            result,
            status 
        FROM crm_ntl_annotation.annotation_results
        WHERE date(
            from_unixtime(
                unix_timestamp(split_part(replace(annotated_at, 'T', ' '), '.', 1), "yyyy-MM-dd HH:mm:ss") + 330*60
            )
        ) >= date('2025-04-08')
    ) rs
    LEFT JOIN crm_ntl_annotation.annotation_data dt
        ON rs.data_id = dt.id
    LEFT JOIN mjolnir_service.users us
        ON us.account_id = rs.annotated_by
    WHERE dt.template_id = 177
),

parsed AS (
    SELECT 
        chat_reference_id,
        annotatedAt,
        annotatedBy,
        result,
        cast(json_parse(result) AS array<json>) AS x
    FROM t
),

data AS (
    SELECT 
        chat_reference_id,
        annotatedBy,
        date(from_unixtime(unix_timestamp(annotatedAt, "yyyy-MM-dd'T'HH:mm:ss") + 330*60)) AS annotated_at_date,
        from_unixtime(unix_timestamp(annotatedAt, "yyyy-MM-dd'T'HH:mm:ss") + 330*60) AS annotated_at_ist,
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'category' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Category",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') LIKE '%subcategory%' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Sub-Category",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'correctint' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Correct interpretation of customer's query / message",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'clarification' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Asked for customer clarification for any response that it could not understand",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'Othertxn' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Reference to correct LAN / transaction ID ",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'correctinfo' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Correct information as per SOP / guidelines",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'hallucination' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "No hallucinations (by GenAI Bot)",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'unnecessary' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "No Unnecessary information",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'latch' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Not latching to voice of customer",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'allqueries' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Addressed all queries of the customer",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'allinfo' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Given all information regarding a query",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'empathy' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Demonstrated understanding of customer emotions and concerns",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'easyunder' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Replies are easy to understand",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'language' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Adherence to customer language preference",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'allmessage' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Did GENAI consider all back-to-back customer messages",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'loop' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Did not go on a loop when customer said Thanks / Ok",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'upload' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Was it able to comprehend the uploaded document?",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'profess' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Professional behaviour with customer",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'openclose' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Opening / Closing",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'cxend' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Did customer end chat after we provided resolution?",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'rep' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Unique responses by the agent",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'closure' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Did GENAI close the ticket?",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'csat' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Closing script adherence",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'escalationagent' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Correctly escalated to an agent",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'escalationnpci' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Correctly escalated to NPCI",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'GENAI' THEN json_format(json_extract(t.x,'$.value.choices')) ELSE null END) AS "Was GENAI Disposition correct?",
        max(CASE WHEN json_extract_scalar(t.x,'$.from_name') = 'remarks' THEN json_format(json_extract(t.x,'$.value.text')) ELSE null END) AS "Remarks"
    FROM parsed p
    LATERAL VIEW explode(x) t AS t
    WHERE date(from_unixtime(unix_timestamp(annotatedAt, "yyyy-MM-dd'T'HH:mm:ss") + 330*60)) = current_date()
    GROUP BY 1,2,3,4
)

SELECT DISTINCT
    a.*,
    b.csat
FROM data a
LEFT JOIN (
    SELECT canonical_ticket_id, AVG(csat) AS csat
    FROM corp_analytics.cx_sub_ticket_data_model_tbl
    GROUP BY canonical_ticket_id
) b
ON a.chat_reference_id = b.canonical_ticket_id;
"""
df = get_sql_df(query)

# Step 3: Connect to the Google Sheet
gsheet_url = "https://docs.google.com/spreadsheets/d/1K74Rj7anxi6z5EKoVQnbCovT2BZMe0eA6NyCdre5Qqs/edit?gid=1595405876#gid=1595405876"
worksheet_name = "Krisha"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

# Step 4: Clear existing data from the worksheet
gs_reader.clear_sheets(worksheet_name, start_row=1, start_column=2)

# Step 5: Write the new DataFrame to the worksheet (including headers)
gs_reader.set_df_in_sheets(worksheet_name, df, include_headers=True, row=1, col=2)